# Desarrollo de Agentes de IA con LangGraph 🤖🛠️

Este notebook está diseñado para el desarrollo práctico de tu curso de **Agentes de Inteligencia Artificial utilizando LangGraph y Google Vertex AI**.

En esta sección nos enfocaremos en uno de los patrones de diseño más importantes: **El Agente con Herramientas (ReAct Agent)**. 

### ¿Qué es el patrón ReAct?
El patrón **Reasoning and Acting (Razonamiento y Acción)** permite a un modelo de lenguaje interactuar con el mundo exterior mediante el uso de herramientas (como buscadores web, calculadoras, consultas a bases de datos, etc.). El agente decide qué herramienta usar, analiza el resultado y decide el siguiente paso de manera cíclica hasta llegar a la respuesta final.

### Paso 1: Instalación de Dependencias 📦
Instalamos las librerías necesarias de forma silenciosa para asegurar un entorno de trabajo limpio.

In [1]:
# Instalar dependencias esenciales de forma silenciosa
!pip install -U -q langchain langgraph google-generativeai langchain-google-genai langchain-community langchain-google-vertexai arxiv >/dev/null 2>&1

### Paso 2: Importaciones y Configuración del Entorno ⚙️
Importamos las librerías principales de LangChain y LangGraph y desactivamos las advertencias para que todas las salidas de código se muestren de forma limpia y profesional.

In [2]:
import os
import warnings

# Supresión global de warnings - Debe ejecutarse ANTES de importar otras librerías
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["PYTHONWARNINGS"] = "ignore"

# Ahora importamos google.generativeai de manera segura
import google.generativeai as genai
from langchain_core._api.deprecation import LangChainDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

# Importaciones de LangChain, Vertex AI y LangGraph
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_core.tools import tool
from langchain_google_vertexai import ChatVertexAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# --- Configuración de Autenticación (Google Colab vs. Google Cloud ADC) ---
try:
    # Intentamos cargar los secretos si estamos ejecutando en Google Colab
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
    os.environ["GOOGLE_API_KEY"] = api_key
    genai.configure(api_key=api_key)
    print("🔑 Autenticado mediante Colab Userdata API Key (Entorno: Google Colab).")
except (ModuleNotFoundError, ImportError):
    # Fuera de Colab (ej. Vertex AI Workbench), el entorno usa de forma nativa ADC (Application Default Credentials)
    print("☁️ Autenticado mediante Application Default Credentials (ADC) de Google Cloud (Entorno local / Workbench).")

print("⚡ Entorno configurado e importaciones listas.")

☁️ Autenticado mediante Application Default Credentials (ADC) de Google Cloud (Entorno local / Workbench).
⚡ Entorno configurado e importaciones listas.


### Paso 3: Instanciación del Modelo Gemini 🚀
En este paso configuramos e instanciamos el modelo **Gemini 2.5 Pro** utilizando la librería de LangChain para Google Generative AI, definimos una plantilla de prompt simple y probamos la conexión invocando una cadena básica.

Adicionalmente, definiremos las herramientas necesarias para los pasos posteriores del agente.

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
import os
from dotenv import load_dotenv

# Configuración del modelo
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

# Definición del prompt template
modelo_de_prompt = PromptTemplate(
    template="Explica de forma clara y detallada el siguiente tema: {tema}",
    input_variables=["tema"]
)

# Creación de la cadena
cadena = modelo_de_prompt | llm | StrOutputParser()

# Ejecución
respuesta = cadena.invoke({"tema": "educación"})
print(respuesta)

# --- Configuración de la API Key de Tavily ---
# Cargamos de forma segura las variables de entorno privadas
load_dotenv("/home/jupyter/.gemini/.env")

try:
    from google.colab import userdata
    # Si estamos en Colab, obtenemos la clave desde los secretos de Colab
    os.environ["TAVILY_API_KEY"] = userdata.get("tavily")
except (ModuleNotFoundError, ImportError):
    # En entornos locales o Vertex AI Workbench, se carga automáticamente desde .env
    pass

# --- Definición de Herramientas para el Agente ---

@tool
def busca_web(query: str) -> list:
    """Realiza una búsqueda en la web sobre un tema específico"""
    tavily_search = TavilySearchResults(
        max_results=2,
        search_depth="advanced",
        max_tokens=1000
    )
    resultado_busca = tavily_search.invoke(query)
    return resultado_busca

@tool
def multiplicar(a: int, b: int) -> int:
    """Multiplica dos números enteros y devuelve el resultado. Use esta herramienta para cualquier operación de multiplicación."""
    return a * b

# Agrupamos las herramientas que usará el agente
tools = [busca_web, multiplicar]


¡Claro que sí! Aquí tienes una explicación clara y detallada sobre el tema de la **educación**, desglosada en sus componentes más importantes para una comprensión completa.

---

### **La Educación: Un Pilar Fundamental para el Desarrollo Humano y Social**

La educación es un concepto mucho más amplio que simplemente ir a la escuela. Es un proceso complejo y multidimensional a través del cual las personas adquieren conocimientos, habilidades, valores, creencias y hábitos. Su objetivo final es el desarrollo integral del individuo y el progreso de la sociedad.

Podemos entender la educación como el conjunto de herramientas que se le entregan a una persona para que pueda comprender el mundo, interactuar con él de manera efectiva y transformarlo para mejor.

---

### **1. Los Propósitos y Objetivos de la Educación**

La educación no tiene un único propósito, sino varios que se complementan entre sí:

*   **Desarrollo Personal y Cognitivo:** Ayuda a las personas a desarrollar el pensamiento

### Paso 4: Definición del Estado del Grafo (State) 📝
En LangGraph, los agentes se modelan como grafos de estado. El **Estado** representa la memoria del agente durante el flujo y se comparte entre todos los nodos.

Usaremos un esquema de estado que almacena una lista de mensajes. La función `add_messages` indica a LangGraph que agregue los nuevos mensajes a la lista existente en lugar de sobrescribirlos.

In [4]:
class AgentState(TypedDict):
    # La lista de mensajes se irá acumulando
    messages: Annotated[list, add_messages]

### Paso 5: Nodos del Grafo (Nodes) 🧠
Los nodos representan unidades de procesamiento o cómputo. Crearemos dos nodos principales:
1. **Model Node (`call_model`)**: Invoca a Gemini vinculando las herramientas disponibles.
2. **Tool Node (`ToolNode`)**: Ejecuta la herramienta solicitada por el modelo. Usaremos el componente preconstruido `ToolNode` de LangGraph para mayor simplicidad.

In [5]:
# Inicializar Gemini 2.5 Flash
model = ChatVertexAI(model_name="gemini-2.5-flash")

# Vincular las herramientas al modelo
model_with_tools = model.bind_tools(tools)

# Nodo que llama al modelo de lenguaje
def call_model(state: AgentState):
    messages = state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

# Nodo que ejecuta las herramientas
tool_node = ToolNode(tools)

### Paso 6: Enrutamiento Condicional (Conditional Edges) 🔀
El enrutamiento condicional decide el flujo basándose en el estado actual.

Después de que el modelo responde, verificamos si ha solicitado llamar a alguna herramienta (tool calls). 
- Si solicitó llamar a una herramienta, dirigimos el flujo hacia el nodo de herramientas (`tools`).
- Si no solicitó herramientas, el agente ha terminado su razonamiento y finalizamos el flujo (`__end__`).

In [6]:
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    messages = state["messages"]
    last_message = messages[-1]
    
    # Si el modelo solicitó llamar a alguna herramienta
    if last_message.tool_calls:
        return "tools"
    
    # De lo contrario, terminamos la ejecución
    return "__end__"

### Paso 7: Construcción y Compilación del Grafo 🕸️
Ahora conectamos los nodos y bordes en nuestro `StateGraph` y lo compilamos para crear el agente ejecutable.

In [7]:
# Inicializar el grafo de estado
workflow = StateGraph(AgentState)

# Agregar los nodos al grafo
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

# Establecer el punto de entrada
workflow.add_edge(START, "agent")

# Agregar borde condicional desde el agente para decidir si usa herramientas o termina
workflow.add_conditional_edges(
    "agent",
    should_continue,
)

# Conectar el nodo de herramientas de regreso al agente
workflow.add_edge("tools", "agent")

# Compilar el grafo
app = workflow.compile()
print("🎉 ¡Grafo de LangGraph compilado exitosamente!")

🎉 ¡Grafo de LangGraph compilado exitosamente!


### Paso 8: Invocación de nuestro Agente con Herramientas 🤖🚀
¡Hagamos una prueba! Le haremos una pregunta matemática que requiera multiplicar para obligar al agente a usar la herramienta que le definimos.

In [8]:
# Definir la pregunta de prueba
pregunta = "Hola, ¿cuánto es 143 multiplicado por 25?"

# Ejecutar el agente e imprimir el flujo paso a paso
inputs = {"messages": [HumanMessage(content=pregunta)]}

print("--- Flujo de Ejecución del Agente ---")
for chunk in app.stream(inputs, stream_mode="values"):
    last_msg = chunk["messages"][-1]
    last_msg.pretty_print()
    print("="*50)

--- Flujo de Ejecución del Agente ---
================================ Human Message =================================

Hola, ¿cuánto es 143 multiplicado por 25?


================================== Ai Message ==================================
Tool Calls:
  multiplicar (151befee-22a8-42c5-aa05-8507c8d58384)
 Call ID: 151befee-22a8-42c5-aa05-8507c8d58384
  Args:
    a: 143.0
    b: 25.0
================================= Tool Message =================================
Name: multiplicar

3575


================================== Ai Message ==================================

El resultado de multiplicar 143 por 25 es 3575.
